<a href="https://colab.research.google.com/github/dcmacedo/google_colab/blob/main/Lotofacil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importe as Bibliotecas Python
No primeiro bloco de código, carregue as ferramentas de manipulação de dados e aprendizado de máquina:

In [29]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## Carregue e Trate os Dados Históricos
Importe os resultados passados contendo os concursos e as 15 dezenas sorteadas (Bola1 a Bola15):

In [30]:
# Carregar o arquivo Excel ou CSV
df = pd.read_excel('Lotofácil.xlsx') # substitua pelo nome do seu arquivo

# Definir colunas das 15 dezenas sorteadas
colunas_bolas = [f'Bola{i}' for i in range(1, 16)]

# Ordenar por concurso
df = df.sort_values('Concurso').reset_index(drop=True)
print(f"Total de concursos carregados: {len(df)}")

Total de concursos carregados: 3800


## Extração de Métrica Estatísticas (Feature Engineering)
Para alimentar o modelo de IA, crie variáveis estatísticas com base nos padrões históricos mais frequentes:
1. Paridade (Pares x Ímpares): a maioria dos concursos possui de 7 a 8 pares.
2. Números Primos: contagem dos primos (2, 3, 5, 7, 11, 13, 17, 19, 23) — normalmente entre 4 e 6 por sorteio.
3. Soma Total: a soma das 15 dezenas concentrada na faixa de 180 a 220.
4. Repetição do Concurso Anterior: média de 8 a 10 dezenas repetidas do sorteio imediatamente anterior.

In [31]:
primos_set = {2, 3, 5, 7, 11, 13, 17, 19, 23}

def extrair_caracteristicas(df):
    features = []
    vals = df[colunas_bolas].to_numpy()

    for i in range(len(vals)):
        linha = vals[i]
        pares = sum(1 for n in linha if n % 2 == 0)
        impares = 15 - pares
        primos = sum(1 for n in linha if n in primos_set)
        soma = sum(linha)

        # Repetidos do concurso anterior
        repetidos = 0
        if i > 0:
            repetidos = len(set(linha).intersection(set(vals[i-1])))

        features.append({
            'Pares': pares,
            'Ímpares': impares,
            'Primos': primos,
            'Soma': soma,
            'Repetidos': repetidos
        })

    return pd.DataFrame(features)

df_features = extrair_caracteristicas(df)

## Construção e Treinamento do Modelo de IA
Podemos treinar um modelo de Random Forest (Floresta Aleatória) ou Classificador Logístico para pontuar a probabilidade de uma determinada combinação atender aos padrões equilibrados de um sorteio real:

In [32]:
# Criar rótulo target: 1 se a combinação atender aos padrões ideais, 0 caso contrário
# Padrão ideal: 7-8 pares, 4-6 primos, soma entre 180 e 220, 8-10 repetidos
df_features['Ideal'] = (
    (df_features['Pares'].between(7, 8)) &
    (df_features['Primos'].between(4, 6)) &
    (df_features['Soma'].between(180, 220)) &
    (df_features['Repetidos'].between(8, 10))
).astype(int)

X = df_features[['Pares', 'Ímpares', 'Primos', 'Soma', 'Repetidos']]
y = df_features['Ideal']

# Divisão em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Treinar o Modelo de IA
modelo_ia = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_ia.fit(X_train, y_train)

print(f"Acurácia do modelo em dados de teste: {modelo_ia.score(X_test, y_test) * 100:.2f}%")

Acurácia do modelo em dados de teste: 100.00%


## Como Funciona o $K$-Fold (com $K=5$)
Os dados são divididos aleatoriamente em 5 partes (dobras/folds) iguais de 20% cada.

O modelo roda 5 rodadas de treinamento:
1. Rodada 1: Treina nas partes 1, 2, 3 e 4; testa na parte 5.
2. Rodada 2: Treina nas partes 1, 2, 3 e 5; testa na parte 4.... e assim por diante até testar todas as partes.

No final, calculamos a média e o desvio padrão da pontuação de todas as rodadas.

Vantagem: Todo o histórico de concursos é testado em algum momento, garantindo que a acurácia medida seja real e não fruto de "sorte" no embaralhamento inicial.

In [33]:
from sklearn.model_selection import KFold, cross_val_score

# 1. Configurar o K-Fold (ex: 5 divisões com embaralhamento)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. Executar a Validação Cruzada sobre todo o conjunto de dados (X e y)
scores = cross_val_score(modelo_ia, X, y, cv=kf, scoring='accuracy')

# 3. Exibir os resultados de cada rodada
print("--- Resultados da Validação Cruzada (5-Fold) ---")
for i, score in enumerate(scores, 1):
    print(f"Rodada {i}: Acurácia = {score * 100:.2f}%")

print("\n--- Métrica Final ---")
print(f"🎯 Acurácia Média: {scores.mean() * 100:.2f}%")
print(f"📊 Desvio Padrão (Estabilidade): ± {scores.std() * 100:.2f}%")

--- Resultados da Validação Cruzada (5-Fold) ---
Rodada 1: Acurácia = 100.00%
Rodada 2: Acurácia = 100.00%
Rodada 3: Acurácia = 100.00%
Rodada 4: Acurácia = 100.00%
Rodada 5: Acurácia = 100.00%

--- Métrica Final ---
🎯 Acurácia Média: 100.00%
📊 Desvio Padrão (Estabilidade): ± 0.00%


## Gerador e Otimizador de Apostas com Validação por IA
Com o modelo treinado, podemos gerar apostas aleatórias e usar a IA para filtrar apenas as que possuem a melhor pontuação de equilíbrio:

In [34]:
def gerar_e_filtrar_aposta(modelo, ultimo_concurso_dezenas, max_tentativas=1000):
    for _ in range(max_tentativas):
        # Selecionar 15 dezenas aleatórias de 1 a 25
        aposta = sorted(np.random.choice(range(1, 26), size=15, replace=False))

        # Calcular estatísticas da aposta candidata
        pares = sum(1 for n in aposta if n % 2 == 0)
        impares = 15 - pares
        primos = sum(1 for n in aposta if n in primos_set)
        soma = sum(aposta)
        repetidos = len(set(aposta).intersection(set(ultimo_concurso_dezenas)))

        dados_aposta = pd.DataFrame([{
            'Pares': pares,
            'Ímpares': impares,
            'Primos': primos,
            'Soma': soma,
            'Repetidos': repetidos
        }])

        # Previsão da IA (1 = aprovada pelo padrão)
        pred = modelo.predict(dados_aposta)
        if pred == 1:
            return aposta, dados_aposta.iloc[0].to_dict()

    return None, None

# Testar a geração com base no último sorteio
ultimo_sorteio = df[colunas_bolas].iloc[-1].tolist()
aposta_otimizada, stats = gerar_e_filtrar_aposta(modelo_ia, ultimo_sorteio)

# Linha formatada
dezenas_formatadas = " - ".join([f"{int(n):02d}" for n in aposta_otimizada])
print(f"📋 Jogo: {dezenas_formatadas}\n")

print(f"📊 Estatísticas da Aposta: {stats}")

📋 Jogo: 01 - 03 - 04 - 05 - 06 - 07 - 10 - 11 - 13 - 14 - 18 - 20 - 22 - 23 - 25

📊 Estatísticas da Aposta: {'Pares': 7, 'Ímpares': 8, 'Primos': 6, 'Soma': 182, 'Repetidos': 8}


## Extrair o Dia da Semana da Coluna de Datas
No Pandas, a função .dt.dayofweek converte uma data em um número de 0 a 6 (onde 0 = Segunda-feira, 1 = Terça-feira, ..., 5 = Sábado).

In [39]:
# Converter a coluna para o formato de data do Pandas
df['Data Sorteio'] = pd.to_datetime(df['Data Sorteio'], errors='coerce')

# Mapear o dia da semana (0 = Segunda, 1 = Terça, ..., 5 = Sábado)
dias_map = {
    0: 'Segunda-feira',
    1: 'Terça-feira',
    2: 'Quarta-feira',
    3: 'Quinta-feira',
    4: 'Sexta-feira',
    5: 'Sábado',
    6: 'Domingo'
}

df['Dia_Semana_Num'] = df['Data Sorteio'].dt.dayofweek
df['Dia_Semana_Nome'] = df['Dia_Semana_Num'].map(dias_map)

## Criar um Menu Seletor Interativo no Colab
Você pode usar os formulários interativos do Colab para escolher o dia antes de rodar o modelo:

In [40]:
# Menu suspenso para escolher o dia da semana
dia_selecionado = "Sexta-feira" #@param ["Segunda-feira", "Terça-feira", "Quarta-feira", "Quinta-feira", "Sexta-feira", "Sábado"]

print(f"🗓️ Dia selecionado para análise: {dia_selecionado}")

🗓️ Dia selecionado para análise: Sexta-feira


## Filtrar o Histórico e Recalcular as Estatísticas do Dia
Filtre a base histórica para manter apenas os sorteios daquele dia da semana e calcule as dezenas mais frequentes e os padrões específicos do dia escolhido:

In [41]:
# 1. Filtrar o DataFrame apenas para os concursos do dia escolhido
df_dia = df[df['Dia_Semana_Nome'] == dia_selecionado].copy()

print(f"Total de concursos realizados em {dia_selecionado}s: {len(df_dia)}")

# 2. Recomputar a frequência das dezenas apenas para este dia da semana
dezenas_dia = df_dia[colunas_bolas].values.flatten()
freq_dezenas_dia = pd.Series(dezenas_dia).value_counts().sort_index()

# Exibir as 5 dezenas "mais quentes" especificamente das segundas/terças/etc.
dezenas_mais_frequentes_dia = freq_dezenas_dia.sort_values(ascending=False).head(15).index.tolist()
print(f"🔥 Dezenas mais sorteadas em {dia_selecionado}s: {sorted(dezenas_mais_frequentes_dia)}")

Total de concursos realizados em Sexta-feiras: 720
🔥 Dezenas mais sorteadas em Sexta-feiras: [2, 5, 7, 8, 10, 11, 14, 15, 17, 18, 19, 20, 22, 24, 25]


## Treinar a IA e Gerar a Aposta Otimizada do Dia
Agora, ajuste a função de geração de jogos para consultar as estatísticas e a frequência do dia selecionado:

In [42]:
def gerar_aposta_por_dia(modelo, df_historico_dia, dezenas_mais_frequentes, max_tentativas=1000):
    primos_set = {2, 3, 5, 7, 11, 13, 17, 19, 23}

    # Pegar as dezenas do último concurso ocorrido nesse dia da semana
    ultimo_concurso_dia = df_historico_dia[colunas_bolas].iloc[-1].tolist()

    for _ in range(max_tentativas):
        # Gera uma combinação valorizando as dezenas mais frequentes daquele dia da semana
        # Ex: escolhe 10 dezenas do grupo "quente" do dia e 5 do restante
        candidatas_quentes = np.random.choice(dezenas_mais_frequentes, size=10, replace=False)
        outras_dezenas = [d for d in range(1, 26) if d not in candidatas_quentes]
        candidatas_outras = np.random.choice(outras_dezenas, size=5, replace=False)

        aposta = sorted(list(candidatas_quentes) + list(candidatas_outras))

        # Extrair métricas
        pares = sum(1 for n in aposta if n % 2 == 0)
        impares = 15 - pares
        primos = sum(1 for n in aposta if n in primos_set)
        soma = sum(aposta)
        repetidos = len(set(aposta).intersection(set(ultimo_concurso_dia)))

        dados_aposta = pd.DataFrame([{
            'Pares': pares,
            'Ímpares': impares,
            'Primos': primos,
            'Soma': soma,
            'Repetidos': repetidos
        }])

        # Testar no modelo de IA
        if modelo.predict(dados_aposta) == 1:
            return aposta, dados_aposta.iloc[0].to_dict()

    return None, None

# Executar a geração para o dia escolhido
aposta_dia, estatisticas_aposta = gerar_aposta_por_dia(modelo_ia, df_dia, dezenas_mais_frequentes_dia)

print(f"\n🎯 Aposta recomendada pela IA para {dia_selecionado}:\n")

# Linha formatada
dezenas_formatadas = " - ".join([f"{int(n):02d}" for n in aposta_dia])
print(f"📋 Jogo: {dezenas_formatadas}\n")

# Visualização no formato de Volante 5x5
print("--- Volante da Lotofácil ---")
aposta_set = set(int(n) for n in aposta_dia)

for i in range(1, 26):
    if i in aposta_set:
        print(f"[{i:02d}]", end=" ")
    else:
        print(" -- ", end=" ")

    # Quebra de linha a cada 5 números
    if i % 5 == 0:
        print()

print("\n📊 Métricas da combinação gerada:")
print(estatisticas_aposta)


🎯 Aposta recomendada pela IA para Sexta-feira:

📋 Jogo: 01 - 02 - 05 - 07 - 08 - 10 - 11 - 14 - 15 - 17 - 18 - 19 - 20 - 21 - 22

--- Volante da Lotofácil ---
[01] [02]  --   --  [05] 
 --  [07] [08]  --  [10] 
[11]  --   --  [14] [15] 
 --  [17] [18] [19] [20] 
[21] [22]  --   --   --  

📊 Métricas da combinação gerada:
{'Pares': 7, 'Ímpares': 8, 'Primos': 6, 'Soma': 190, 'Repetidos': 9}
